In [1]:
import sys
import os
import torch
import glob

from datasets import load_dataset
from torch_geometric.datasets import QM9
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import rdmolfiles

import numpy as np
import pandas as pd
import ast

In [3]:
from datasets import load_dataset

ds = load_dataset("yairschiff/qm9")

# Use `ds['canonical_smiles']` from `rdkit` as inputs.


In [7]:
df_all = ds['train'].to_pandas()
df_all.to_csv(r'C:\Users\chris\PycharmProjects\ML_project\data\QM9\QM9_complete.csv', index=False)

In [ ]:
df_all.to_csv(r'C:\Users\chris\PycharmProjects\ML_project\data\QM9\QM9_complete.csv', index=False)

In [ ]:
pyg_dataset = []

# Atom-Symbol zu Ordnungszahl Mapping
symbol_to_z = {'H': 1, 'C': 6, 'N': 7, 'O': 8, 'F': 9}

print("Konvertierte in PyTorch Geometric Graphen...")
for idx, (_, row) in enumerate(df_all.iterrows()):
    try:
        # 1. Spalten auslesen
        atomic_symbols = row['atomic_symbols']
        pos_raw = row['pos']

        # Sicherstellen, dass Strings (Text) wieder in echte Listen umgewandelt werden
        if isinstance(atomic_symbols, str):
            atomic_symbols = ast.literal_eval(atomic_symbols)
        if isinstance(pos_raw, str):
            pos_raw = ast.literal_eval(pos_raw)

        # Falls es ein NumPy-Objekt-Array ist, in eine normale Python-Liste zwingen
        if isinstance(pos_raw, np.ndarray):
            pos_raw = pos_raw.tolist()

        # Konvertierung zu reinen float-Werten erzwingen (behebt den NumPy-Object Fehler)
        pos_list = [[float(coord) for coord in atom_pos] for atom_pos in pos_raw]

        # a) Atom-Typen (z) extrahieren und in numerische Features (x) umwandeln
        z = torch.tensor([symbol_to_z[sym] for sym in atomic_symbols], dtype=torch.long)
        x = z.view(-1, 1)

        # b) 3D-Koordinaten (pos) laden (Jetzt garantiert ohne Typ-Fehler)
        pos = torch.tensor(pos_list, dtype=torch.float)

        # c) Zielwerte (y) extrahieren (HOMO und LUMO)
        y = torch.tensor([[float(row['homo']), float(row['lumo'])]], dtype=torch.float)

        # d) Kanten bestimmen (Radius-Graph)
        from torch_geometric.nn import radius_graph
        edge_index = radius_graph(pos, r=2.0, max_num_neighbors=35)

        name = f"gdb_{idx + 1}"

        data = Data(x=x, pos=pos, edge_index=edge_index, y=y, name=name)
        pyg_dataset.append(data)

    except Exception as e:
        print(f"Fehler bei Zeile {idx}: {e}")
        continue

print(f"Fertig! {len(pyg_dataset)} Graphen erfolgreich erstellt.")

# Jetzt können Sie sofort den DataLoader für das GNN-Training nutzen:
loader = DataLoader(pyg_dataset, batch_size=32, shuffle=True)